# Adult 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Adult 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/adult_bttwd.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/adult_bttwd.yaml
运行消融实验： False


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/adult_bttwd.yaml
标准输出：
【INFO】【2026-05-18 19:27:57】【数据加载完毕】样本数=32561，特征数=14，正类比例=0.24
【INFO】【2026-05-18 19:27:57】【数据加载】24720 条标签无法映射，未指定负类且未开启 dropna_target，已按 0 处理
【INFO】【2026-05-18 19:27:57】【数据加载】标签列 income 已处理完成：dropna_target=False, 丢弃样本=0, 最终样本数=32561, 正类比例=24.08%
【INFO】【2026-05-18 19:27:57】【数据集信息】名称=adult，样本数=32561，目标列=income，正类比例=24.08%
【INFO】【2026-05-18 19:27:57】【预处理】连续特征=6个，类别特征=8个
【INFO】【2026-05-18 19:27:57】【预处理】编码后维度=100
【INFO】【2026-05-18 19:27:57】【governance】adult_bttwd 第 1/5 折
【INFO】【2026-05-18 19:27:58】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-18 19:27:58】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-18 19:28:11】【governance】weak bucket resolver：strong=24，weak=45
【INFO】【2026-05-18 19:28:11】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-18 19:28:28】【governance】adult_bttwd 第 2/5 折
【INFO】【2026-05-18 19:28:28】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-18 19:28:28】【桶树】已为样本生成桶ID，共 48 个组合
【INFO】【2026-05-18 19:28:41】【governa

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,adult_bttwd,0.150904,0.82534,0.72786,0.865238,0.153627,0.809178,0.716432,0.867142,1.0,...,0.967213,0.089888,0.910112,0.032787,0.933333,0.406667,0.526667,0.877325,137.0,5.0


Reading: outputs\governance\full\adult_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,adult_bttwd,1,0.151528,0.823971,0.726820,0.865193,0.158376,0.805480,0.711564,0.865193,...,0.875000,0.214286,0.785714,0.125000,0.818182,0.363636,0.454545,0.660714,9.35,1.45
1,adult_bttwd,2,0.148603,0.828370,0.731889,0.867015,0.150799,0.801463,0.712060,0.869472,...,0.954545,0.031250,0.968750,0.045455,0.962963,0.407407,0.555556,0.923295,52.70,2.90
2,adult_bttwd,3,0.151152,0.823175,0.727273,0.866400,0.151743,0.804138,0.713235,0.868243,...,1.000000,0.000000,1.000000,0.000000,1.000000,0.531915,0.468085,1.000000,18.70,0.00
3,adult_bttwd,4,0.152273,0.825087,0.725538,0.863022,0.154845,0.820421,0.723618,0.864865,...,1.000000,0.333333,0.666667,0.000000,0.733333,0.200000,0.533333,0.666667,6.80,0.00
4,adult_bttwd,5,0.150967,0.826098,0.727778,0.864558,0.152373,0.814387,0.721683,0.867936,...,0.976190,0.106061,0.893939,0.023810,0.925926,0.388889,0.537037,0.870130,50.15,1.45


Sample records: outputs\governance\full\adult_bttwd\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


RUN_ABLATION=False，跳过无 CP 消融。


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


RUN_ABLATION=False，跳过无渐进更新消融。
